# Phase 2 (protocol 06 / 06a) — verdicts from saved results

Recomputes the Phase 2 verdicts (Q1, H5–H8), the quality-versus-minting table and figure 8 from the
downloaded Phase 1 (`experiments/modal-m3/runs`) and Phase 2 (`experiments/modal-p2/runs`) run directories.
No GPU and no training. The Phase 1 verdicts are recomputed with the Phase 2 arms present and must be unchanged.

Later correction: every "Dense" here is the dense head without an output bias. The prior-matched control
(protocol 08, `results/phase4/prior_verdict.json`) gave Dense the same log-unigram bias that the KAS-U heads have; it
then beat KAS-U64 on every seed, so the KAS-U64 lead shown below does not survive information parity.

In [1]:
import json, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "results" / "phase2"
RUNS = [str(ROOT / "experiments/modal-m3/runs"), str(ROOT / "experiments/modal-p2/runs")]

## 1. Joint analysis of Phase 1 + Phase 2 arms (same code and bootstraps as Phase 1)

In [2]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "final_analysis.py"), "--runs", *RUNS,
                    "--prefix", "m3", "p2", "--split", "test", "--out", str(OUT)], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-2500:] or r.stderr[-2500:])
A = json.load(open(OUT / "analysis_m3_p2_test.json"))
P1 = json.load(open(ROOT / "results/final/analysis_m3_test.json"))
for h in ("H1", "H2", "H3", "H4"):
    assert A[h]["verdict"] == P1[h]["verdict"], h
print("Phase 1 verdicts unchanged:", {h: A[h]["verdict"] for h in ("H1", "H2", "H3", "H4")})
for arm, v in A["arms"].items():
    print(f"{arm:18s} {v['bpb_mean']:.4f}  seeds {[round(x, 4) for x in v['bpb_per_seed']]}  head {v['head_params']:,}  V-dep {v['head_V_dependent']:,}")

-0": [
   1.939758849337148,
   0.0003078839521506449
  ],
  "KAS-G": [
   1.5138022752228641,
   0.008570235972703344
  ],
  "KAS-G-occ": [
   1.5522132007442941,
   0.0006426045155298876
  ],
  "KAS-G-shuf": [
   1.5278782600012273,
   0.0028267478735242157
  ],
  "KAS-P": [
   1.630780687984679,
   0.00013835484706525847
  ],
  "KAS-U16": [
   1.4612712354119408,
   0.0009924352522090452
  ],
  "KAS-U16 (c_zero)": [
   1.5829896785226873,
   0.00042851710831559586
  ],
  "KAS-U16-drop25": [
   1.6346950107200269,
   0.006849679587692448
  ],
  "KAS-U16-dropU25": [
   1.4843530199998463,
   0.0035112362963403143
  ],
  "KAS-U64": [
   1.4145933887818085,
   0.004315857373428322
  ]
 },
 "verdicts": {
  "H1": "SUPPORTED",
  "H2": "SUPPORTED",
  "H3": "INCONCLUSIVE",
  "H4": "REJECTED"
 },
 "fairness": {
  "distinct_train_orders": 1,
  "distinct_total_steps": [
   1526
  ],
  "distinct_data_manifests": 1,
  "seeds_per_arm": {
   "Dense": [
    1,
    2
   ],
   "KAS-0": [
    1,
    2


## 2. Phase 2 verdicts

In [3]:
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "phase2_analysis.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:] or r.stderr[-3000:])
V = json.load(open(OUT / "phase2_verdicts.json"))
def ci(x): return f"{x['point']:+.4f} [{x['ci95'][0]:+.4f}, {x['ci95'][1]:+.4f}]"
for k, v in A["bpb_differences"].items():
    if v and ("drop" in k or "occ" in k or "U64" in k):
        print(f"{k:34s} {ci(v)}")

{
 "bpb": {
  "KAS-U16": 1.4612712354119408,
  "KAS-G": 1.5138022752228641,
  "KAS-P": 1.630780687984679,
  "Dense": 1.442741495665308,
  "KAS-G-shuf": 1.5278782600012273,
  "KAS-U16-drop25": 1.6346950107200269,
  "KAS-U16-dropU25": 1.4843530199998463,
  "KAS-G-occ": 1.5522132007442941,
  "KAS-U64": 1.4145933887818085
 },
 "verdicts": {
  "H7_report_generator_form": "DOES NOT REPLICATE",
  "H8_rank64": "TRAINS",
  "H5": {
   "KAS-U16-drop25": "SUPPORTED",
   "KAS-U16-dropU25": "SUPPORTED"
  },
  "H6": {
   "KAS-U16-drop25": {
    "vs_KAS-G": "DOMINATED",
    "vs_Dense": "INCONCLUSIVE"
   },
   "KAS-U16-dropU25": {
    "vs_KAS-G": "TRADE-OFF",
    "vs_Dense": "INCONCLUSIVE"
   }
  }
 }
}

KAS-U16-drop25 - KAS-U16           +0.1734 [+0.1680, +0.1789]
KAS-U16-drop25 - KAS-G             +0.1209 [+0.1130, +0.1296]
KAS-U16-drop25 - Dense             +0.1920 [+0.1859, +0.1983]
KAS-P - KAS-U16-drop25             -0.0039 [-0.0098, +0.0021]
KAS-P - KAS-G-occ                  +0.0786 [+0.0773, +0

## 3. Quality versus minting cost, every arm

In [4]:
T = V["quality_vs_minting"]["table"]
print(f"{'arm':16s} {'rule':14s} {'bpb':>8s} {'net x1e3':>9s}")
for arm, row in sorted(T.items(), key=lambda kv: kv[1]["in_vocab_test_bpb"]):
    print(f"{arm:16s} {row['rule_by_net_dev']:14s} {row['in_vocab_test_bpb']:8.4f} {row['net_minting_test_bpb']*1e3:+9.3f}")
print("non-dominated:", V["quality_vs_minting"]["non_dominated"])
print("dominance:", V["quality_vs_minting"]["dominance"])

arm              rule                bpb  net x1e3
KAS-U64          inherit_prior    1.4146    +7.788
Dense            inherit          1.4427    +1.325
KAS-U16          inherit_prior    1.4613    +2.607
KAS-U16-dropU25  inherit          1.4844    +0.739
KAS-G            gen_count        1.5138    +0.400
KAS-G-occ        gen_inherit      1.5522    +0.403
KAS-P            inherit          1.6308    +0.317
KAS-U16-drop25   inherit          1.6347    +0.744
non-dominated: ['KAS-U16-dropU25', 'KAS-U64', 'KAS-G', 'KAS-P', 'Dense']
dominance: ['KAS-U16-dropU25 dominates KAS-U16-drop25', 'KAS-G dominates KAS-U16-drop25', 'KAS-G dominates KAS-G-occ', 'KAS-G-occ dominates KAS-U16-drop25', 'KAS-P dominates KAS-U16-drop25', 'Dense dominates KAS-U16']


## 4. Figure 8

The image output is not stored in this notebook; the figure is `figures/fig8_phase2_gap_and_frontier.png`.

In [5]:
subprocess.run([sys.executable, str(ROOT / "scripts" / "phase2_figures.py")], cwd=ROOT, check=True)
from IPython.display import Image, display
display(Image(filename=str(ROOT / "figures" / "fig8_phase2_gap_and_frontier.png"), width=800))